In [1]:
# Import Libraries
import os
import sys

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, os.pardir))
sys.path.insert(0, parent_dir)

import torch
from PIL import Image
from torchvision import transforms

# import numpy as np
# import matplotlib.pyplot as plt

In [2]:
from modules.retrieval.pipeline import Pipeline

# VGGFACE2 Model
pipeline_vgg = Pipeline(pretrained='vggface2', device='cpu')
pipeline_vgg.process_gallery('../storage/multi_image_gallery', 'vgg')
pipeline_vgg.load_embeddings('vgg')

# CASIA-WEBFACE Model
pipeline_casia = Pipeline(pretrained='casia-webface', device='cpu')
pipeline_casia.process_gallery('../storage/multi_image_gallery', 'casia')
pipeline_casia.load_embeddings('casia')

# Test
probe_image = Image.open('../simclr_resources/probe/Ann_Veneman/Ann_Veneman_0002.jpg')

print("Person Name: Ann Veneman\n")
print("VGGFACE2 Model Results:")
results_vgg = pipeline_vgg.search_gallery(probe_image, k=5)
for result in results_vgg:
    print(f"Match: {result['name']}")

print("\nCASIA-WEBFACE Model Results:")
results_casia = pipeline_casia.search_gallery(probe_image, k=5)
for result in results_casia:
    print(f"Match: {result['name']}")

/Users/sichaoliu/anaconda3/envs/myenv/lib/python3.12/site-packages/facenet_pytorch/models/inception_resnet_v1.py:329: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dic

Person Name: Ann Veneman

VGGFACE2 Model Results:
Match: Ann_Veneman
Match: Ann_Veneman
Match: Ann_Veneman
Match: Hitomi_Soga
Match: Ann_Veneman

CASIA-WEBFACE Model Results:
Match: Liza_Minnelli
Match: Kalpana_Chawla
Match: Carol_Burnett
Match: Oscar_De_La_Hoya
Match: Marco_Antonio_Barrera


In [3]:
def load_test_data(path):
    test_data = []
    valid_extensions = ('.png', '.jpg', '.jpeg')
    
    for person_name in os.listdir(path):
        person_folder = os.path.join(path, person_name)
        
        if os.path.isdir(person_folder):
            for filename in os.listdir(person_folder):
                if filename.lower().endswith(valid_extensions):
                    image_path = os.path.join(person_folder, filename)
                    try:
                        image = Image.open(image_path).convert('RGB')
                        test_data.append((image, person_name))
                    except Exception as e:
                        print(f"Error loading image {filename}: {str(e)}")
    
    return test_data

test_data = load_test_data('../simclr_resources/probe')

In [4]:
def evaluate_model(pipeline, test_data, top_n=1):
    predictions = []
    true_labels = []
    
    for image, label in test_data:
        results = pipeline.search_gallery(image, k=top_n)
        pred = [result['name'] for result in results]
        predictions.append(pred)
        true_labels.append(label)

    # Accuracy, Precision, Recall, and F1 score (weighted average)
    total = len(true_labels)
    labels = set(true_labels)  
    accuracy_total, precision_total, recall_total, f1_total = 0, 0, 0, 0

    for label in labels:
        tp = sum(1 for true, pred in zip(true_labels, predictions) if true in pred and true == label)
        fp = sum(1 for true, pred in zip(true_labels, predictions) if true != label and label in pred)
        fn = sum(1 for true, pred in zip(true_labels, predictions) if true == label and label not in pred)
        tn = sum(1 for true, pred in zip(true_labels, predictions) if true != label and label not in pred)

        accuracy = (tp + tn) / (tp + fp + fn + tn) if (tp + fp + fn + tn) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        weight = sum(1 for t in true_labels if t == label)  
        accuracy_total += accuracy * weight
        precision_total += precision * weight
        recall_total += recall * weight
        f1_total += f1 * weight

    accuracy_weighted = accuracy_total / total
    precision_weighted = precision_total / total
    recall_weighted = recall_total / total
    f1_weighted = f1_total / total
    
    return {
        'accuracy': f'{accuracy_weighted:.4f}',
        'precision': f'{precision_weighted:.4f}',
        'recall': f'{recall_weighted:.4f}',
        'f1': f'{f1_weighted:.4f}',
    }

# metrics_vgg = evaluate_model(pipeline_vgg, test_data, top_n=1)
# metrics_casia = evaluate_model(pipeline_casia, test_data, top_n=1)

# print("VGGFACE2 Model Results (top 1):")
# print(metrics_vgg)

# print("\nCASIA-WEBFACE Model Results (top 1):")
# print(metrics_casia)

In [5]:
# # PLOT CRASHED THE ENVIRONMENT
# def plot_metrics(metrics_vgg, metrics_casia, top_n_values):

In [6]:
top_n_values = [1, 3, 5, 10]  

for top_n in top_n_values:
    metrics_vgg = evaluate_model(pipeline_vgg, test_data, top_n=top_n)
    metrics_casia = evaluate_model(pipeline_casia, test_data, top_n=top_n)
    print(f"\nVGGFACE2 Model Results (top {top_n}):")
    print(metrics_vgg)

    print(f"\nCASIA-WEBFACE Model Results (top {top_n}):")
    print(metrics_casia)


VGGFACE2 Model Results (top 1):
{'accuracy': '0.9990', 'precision': '0.3988', 'recall': '0.5065', 'f1': '0.4293'}

CASIA-WEBFACE Model Results (top 1):
{'accuracy': '0.9982', 'precision': '0.0596', 'recall': '0.0931', 'f1': '0.0679'}

VGGFACE2 Model Results (top 3):
{'accuracy': '0.9976', 'precision': '0.3109', 'recall': '0.6196', 'f1': '0.3785'}

CASIA-WEBFACE Model Results (top 3):
{'accuracy': '0.9963', 'precision': '0.0570', 'recall': '0.1361', 'f1': '0.0710'}

VGGFACE2 Model Results (top 5):
{'accuracy': '0.9961', 'precision': '0.2378', 'recall': '0.6647', 'f1': '0.3161'}

CASIA-WEBFACE Model Results (top 5):
{'accuracy': '0.9945', 'precision': '0.0521', 'recall': '0.1632', 'f1': '0.0685'}

VGGFACE2 Model Results (top 10):
{'accuracy': '0.9922', 'precision': '0.1470', 'recall': '0.7177', 'f1': '0.2207'}

CASIA-WEBFACE Model Results (top 10):
{'accuracy': '0.9899', 'precision': '0.0467', 'recall': '0.2122', 'f1': '0.0642'}


# VGGFace2 vs CASIA-WebFace

The "top-k" setting refers to considering the k highest-ranked matches as potential correct identifications. As k increases, we allow for more potential matches, which can increase the chance of including the correct match but also introduces more potential false positives.

## Performance Metrics in Context

1. **Accuracy**: The proportion of correct predictions (both true positives and true negatives) among the total number of cases examined. In this case, the very high accuracy scores are primarily due to a large number of true negatives, which is common in facial recognition systems where most comparisons do not result in a match.

2. **Precision**: The proportion of correct positive identifications out of all positive identifications made. This is crucial in facial recognition to minimize false positives.

3. **Recall**: The proportion of actual positives that were correctly identified. In facial recognition, this represents the system's ability to find a match when it exists.

4. **F1 Score**: The harmonic mean of precision and recall, providing a single score that balances both metrics.

## Results Analysis

### VGGFace2 Model Performance

| Top-k | Accuracy | Precision | Recall | F1 Score |
|-------|----------|-----------|--------|----------|
| 1     | 0.9990   | 0.3988    | 0.5065 | 0.4293   |
| 3     | 0.9976   | 0.3109    | 0.6196 | 0.3785   |
| 5     | 0.9961   | 0.2378    | 0.6647 | 0.3161   |
| 10    | 0.9922   | 0.1470    | 0.7177 | 0.2207   |

#### Analysis:

1. **Accuracy**: VGGFace2 maintains very high accuracy across all top-k settings, but this is primarily due to the large number of true negatives. The slight decrease in accuracy as k increases (from 0.9990 to 0.9922) suggests a small increase in false positives.

2. **Precision**: Precision decreases significantly as top-k increases, from 39.88% for top-1 to 14.70% for top-10. This indicates that as we consider more potential matches, we introduce more false positives.

3. **Recall**: Recall improves as top-k increases, from 50.65% for top-1 to 71.77% for top-10. This shows that considering more potential matches increases the chance of including the correct match.

4. **F1 Score**: The F1 score decreases as top-k increases, from 0.4293 for top-1 to 0.2207 for top-10, indicating that the gain in recall doesn't compensate for the loss in precision.

### CASIA-WebFace Model Performance

| Top-k | Accuracy | Precision | Recall | F1 Score |
|-------|----------|-----------|--------|----------|
| 1     | 0.9982   | 0.0596    | 0.0931 | 0.0679   |
| 3     | 0.9963   | 0.0570    | 0.1361 | 0.0710   |
| 5     | 0.9945   | 0.0521    | 0.1632 | 0.0685   |
| 10    | 0.9899   | 0.0467    | 0.2122 | 0.0642   |

#### Analysis:

1. **Accuracy**: CASIA-WebFace also shows very high accuracy, but like VGGFace2, this is primarily due to high true negatives. The accuracy decreases slightly as top-k increases, indicating an increase in false positives.

2. **Precision**: Precision is consistently low and decreases as top-k increases, from 5.96% for top-1 to 4.67% for top-10. This suggests a high rate of false positives and comparatively low true positives.

3. **Recall**: Recall improves as top-k increases, from 9.31% for top-1 to 21.22% for top-10, but remains relatively low compared to VGGFace2 due to the low true positives.

4. **F1 Score**: The F1 score remains low across all top-k settings, ranging from 0.0679 to 0.0642, indicating poor overall performance in balancing precision and recall.

## Comparative Analysis

1. **Overall Performance**: VGGFace2 outperforms CASIA-WebFace across all metrics except accuracy, where both models perform similarly due to the high number of true negatives.

2. **Precision vs. Recall Trade-off**: Both models show a trade-off between precision and recall as top-k increases. VGGFace2 manages this trade-off better, maintaining higher precision and recall across all top-k settings.

3. **False Positive Rate**: The decrease in accuracy and precision as top-k increases indicates a rising false positive rate for both models, but this increase is more pronounced in CASIA-WebFace.

4. **True Positive Rate**: VGGFace2 shows a significantly better ability to identify correct matches (higher recall) compared to CASIA-WebFace.

Overall, VGGFace2 demonstrates superior performance compared to CASIA-WebFace, particularly in its ability to balance precision and recall. However, the choice of top-k setting and the interpretation of results should be carefully tuned based on the specific requirements of the application, considering the trade-offs between security, usability, and computational resources. The high accuracy scores, while impressive, should be interpreted cautiously due to the prevalence of true negatives. Focus should be placed on improving precision and recall while maintaining a low false positive rate for real-world application success.

## Implications for Facial Recognition System Design

1. **Model Selection**: VGGFace2 is the superior choice for facial recognition tasks based on these results, offering better precision, recall, and F1 scores across all top-k settings.

2. **Balancing Security and Usability**: The choice of top-k setting involves a trade-off between security (precision) and usability (recall). For high-security applications, a lower top-k with VGGFace2 might be preferred to minimize false positives. For applications where missing a match is more costly, a higher top-k might be used.

3. **False Positive Management**: The decreasing precision as top-k increases highlights the need for additional verification steps in the recognition process, especially for higher top-k settings.

4. **Threshold Tuning**: Given the high accuracy due to true negatives, it's crucial to fine-tune the similarity threshold used for determining matches. This can help balance the trade-off between false positives and false negatives.

5. **Continuous Model Improvement**: The relatively low F1 scores for both models suggest room for improvement. Regular retraining with diverse, high-noise data could help enhance performance.

In [7]:
import numpy as np
from scipy.ndimage import convolve

def apply_noise(image, noise_type, severity):    
    image_array = np.array(image).astype(np.float64)
    noisy_image = np.copy(image_array)

    if noise_type == 'gaussian':
        mean = 0
        sigma = severity * 0.1 
        gauss = np.random.normal(mean, sigma, image_array.shape)
        noisy_image += gauss * 255
        noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)

    elif noise_type == 'speckle':
        noise = np.random.randn(*image_array.shape) * (severity / 10)
        noisy_image += image * noise
        noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)

    elif noise_type == 'motion_blur':
        size = severity * 2

        kernel_motion_blur = np.zeros((size, size), dtype=np.float32)
        kernel_motion_blur[int((size - 1) / 2), :] = np.ones(size)
        kernel_motion_blur /= size

        noisy_image = np.zeros_like(image_array)

        for i in range(image_array.shape[2]): 
            noisy_image[:, :, i] = convolve(image_array[:, :, i], kernel_motion_blur, mode='reflect')

        noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)

    else:
        raise ValueError(f"Unknown noise type: {noise_type}")

    return Image.fromarray(noisy_image)


In [8]:
noise_types = ['motion_blur', 'gaussian', 'speckle']
severities = [1, 5, 10]

def evaluate_noise_impact(pipeline, test_data, noise_type, severity, top_n=1):
    noisy_data = [(apply_noise(img, noise_type, severity), label) for img, label in test_data]
    return evaluate_model(pipeline, noisy_data, top_n=top_n)

vgg_noise_results = {}
casia_noise_results = {}

for noise in noise_types:
    vgg_noise_results[noise] = []
    casia_noise_results[noise] = []
    for severity in severities:
        vgg_perf = evaluate_noise_impact(pipeline_vgg, test_data, noise, severity)
        casia_perf = evaluate_noise_impact(pipeline_casia, test_data, noise, severity)
        vgg_noise_results[noise].append(vgg_perf)
        casia_noise_results[noise].append(casia_perf)

print("\nVGGFACE2 Noise Results:")
for noise, results in vgg_noise_results.items():
    print(f"\nNoise Type: {noise}")
    for severity, metrics in zip(severities, results):
        accuracy = float(metrics['accuracy']) if isinstance(metrics['accuracy'], str) else metrics['accuracy']
        precision = float(metrics['precision']) if isinstance(metrics['precision'], str) else metrics['precision']
        recall = float(metrics['recall']) if isinstance(metrics['recall'], str) else metrics['recall']
        f1 = float(metrics['f1']) if isinstance(metrics['f1'], str) else metrics['f1']

        print(f"  Severity {severity}: Accuracy = {accuracy:.4f}, "
              f"Precision = {precision:.4f}, "
              f"Recall = {recall:.4f}, "
              f"F1 Score = {f1:.4f}")
        
print("\nCASIA-WEBFACE Noise Results:")
for noise, results in casia_noise_results.items():
    print(f"\nNoise Type: {noise}")
    for severity, metrics in zip(severities, results):
        accuracy = float(metrics['accuracy']) if isinstance(metrics['accuracy'], str) else metrics['accuracy']
        precision = float(metrics['precision']) if isinstance(metrics['precision'], str) else metrics['precision']
        recall = float(metrics['recall']) if isinstance(metrics['recall'], str) else metrics['recall']
        f1 = float(metrics['f1']) if isinstance(metrics['f1'], str) else metrics['f1']

        print(f"  Severity {severity}: Accuracy = {accuracy:.4f}, "
              f"Precision = {precision:.4f}, "
              f"Recall = {recall:.4f}, "
              f"F1 Score = {f1:.4f}")


VGGFACE2 Noise Results:

Noise Type: motion_blur
  Severity 1: Accuracy = 0.9990, Precision = 0.3901, Recall = 0.5015, F1 Score = 0.4214
  Severity 5: Accuracy = 0.9988, Precision = 0.2907, Recall = 0.3824, F1 Score = 0.3155
  Severity 10: Accuracy = 0.9982, Precision = 0.0613, Recall = 0.0911, F1 Score = 0.0673

Noise Type: gaussian
  Severity 1: Accuracy = 0.9989, Precision = 0.3589, Recall = 0.4605, F1 Score = 0.3870
  Severity 5: Accuracy = 0.9980, Precision = 0.0135, Recall = 0.0240, F1 Score = 0.0155
  Severity 10: Accuracy = 0.9980, Precision = 0.0001, Recall = 0.0030, F1 Score = 0.0002

Noise Type: speckle
  Severity 1: Accuracy = 0.9990, Precision = 0.3876, Recall = 0.4945, F1 Score = 0.4176
  Severity 5: Accuracy = 0.9984, Precision = 0.1566, Recall = 0.2212, F1 Score = 0.1718
  Severity 10: Accuracy = 0.9981, Precision = 0.0163, Recall = 0.0270, F1 Score = 0.0175

CASIA-WEBFACE Noise Results:

Noise Type: motion_blur
  Severity 1: Accuracy = 0.9982, Precision = 0.0550, Reca

# Noise: VGGFace2 vs CASIA-WebFace

Our analysis considers three noise types (motion blur, Gaussian, and speckle) at three severity levels (1, 5, and 10).

## Noise Types

1. **Motion Blur**: Simulates the effect of camera or subject movement during image capture.
2. **Gaussian Noise**: Represents random variation of brightness or color information in images.
3. **Speckle Noise**: Multiplicative noise that commonly occurs in ultrasound imaging.

## Results Analysis

### VGGFace2 Model Performance

#### Motion Blur

| Severity | Accuracy | Precision | Recall | F1 Score |
|----------|----------|-----------|--------|----------|
| 1        | 0.9990   | 0.3901    | 0.5015 | 0.4214   |
| 5        | 0.9988   | 0.2907    | 0.3824 | 0.3155   |
| 10       | 0.9982   | 0.0613    | 0.0911 | 0.0673   |


Analysis:
- VGGFace2 shows moderate resilience to low levels of motion blur.
- Performance degrades significantly at high blur levels, with precision and recall dropping sharply.
- Even at high blur, the model maintains high accuracy due to correct negative classifications.

#### Gaussian Noise

| Severity | Accuracy | Precision | Recall | F1 Score |
|----------|----------|-----------|--------|----------|
| 1        | 0.9989   | 0.3589    | 0.4605 | 0.3870   |
| 5        | 0.9980   | 0.0135    | 0.0240 | 0.0155   |
| 10       | 0.9980   | 0.0001    | 0.0030 | 0.0002   |

Analysis:
- VGGFace2 is highly sensitive to Gaussian noise.
- Performance and recall drop dramatically with increasing noise severity.
- At high noise levels, the model essentially fails to make correct positive identifications.

#### Speckle Noise

| Severity | Accuracy | Precision | Recall | F1 Score |
|----------|----------|-----------|--------|----------|
| 1        | 0.9990   | 0.3876    | 0.4945 | 0.4176   |
| 5        | 0.9984   | 0.1566    | 0.2212 | 0.1718   |
| 10       | 0.9981   | 0.0163    | 0.0270 | 0.0175   |

Analysis:
- VGGFace2 shows better resilience to speckle noise at lower levels.
- Performance degrades more gradually with increasing speckle noise.
- Still, high levels of speckle noise significantly impair the model's ability to make correct identifications.

### CASIA-WebFace Model Performance

#### Motion Blur

| Severity | Accuracy | Precision | Recall | F1 Score |
|----------|----------|-----------|--------|----------|
| 1        | 0.9982   | 0.0550    | 0.0841 | 0.0622   |
| 5        | 0.9981   | 0.0369    | 0.0541 | 0.0410   |
| 10       | 0.9980   | 0.0059    | 0.0120 | 0.0072   |

Analysis:
- CASIA-WebFace demonstrates a degree of resilience at low motion blur severity, although its overall performance remains not so well.
- Performance degrades further with increasing blur, but the initial performance is already low.

#### Gaussian Noise

| Severity | Accuracy | Precision | Recall | F1 Score |
|----------|----------|-----------|--------|----------|
| 1        | 0.9981   | 0.0354    | 0.0541 | 0.0393   |
| 5        | 0.9980   | 0.0000    | 0.0000 | 0.0000   |
| 10       | 0.9980   | 0.0000    | 0.0010 | 0.0000   |

Analysis:
- CASIA-WebFace is extremely sensitive to Gaussian noise.
- The model essentially fails to make any correct positive identifications.

#### Speckle Noise

| Severity | Accuracy | Precision | Recall | F1 Score |
|----------|----------|-----------|--------|----------|
| 1        | 0.9982   | 0.0529    | 0.0791 | 0.0594   |
| 5        | 0.9980   | 0.0059    | 0.0140 | 0.0070   |
| 10       | 0.9980   | 0.0007    | 0.0050 | 0.0011   |

Analysis:
- CASIA-WebFace shows poor performance even with low levels of speckle noise.
- Performance degrades to near-zero positive identifications at higher noise levels.

## Comparative Analysis

1. **Overall Noise Resilience**: VGGFace2 demonstrates significantly better resilience to all types of noise compared to CASIA-WebFace. This is evident in higher precision, recall, and F1 scores across all noise types and severities.

2. **Noise Type Sensitivity**: 
   - Both models are most sensitive to Gaussian noise, with performance degrading rapidly as noise severity increases.
   - Motion blur has the least impact on both models, especially at lower levels.
   - VGGFace2 exhibits greater resilience to speckle noise at lower levels; however, both models struggle significantly at high severity levels.

3. **Performance Degradation**: 
   - VGGFace2 shows a more gradual degradation in performance as noise severity increases.
   - CASIA-WebFace performance is poor even at low noise levels and quickly approaches zero effective positive identification ability with increased noise.


4. **Accuracy Mislead**: Both models maintain very high accuracy across all noise types and severities. This is misleading and primarily due to correct negative classifications in an imbalanced dataset, highlighting the importance of examining precision, recall, and F1 scores.

While VGGFace2 demonstrates superior performance and better resilience to noise compared to CASIA-WebFace, both models show significant sensitivity to image noise, particularly Gaussian noise. This analysis underscores the critical importance of image quality in facial recognition systems. Additionally, we must carefully consider environmental factors and implement appropriate fallback mechanisms to maintain security and usability in challenging conditions.

## Implications for Facial Recognition System Design

1. **Model Selection**: VGGFace2 is clearly superior for facial recognition tasks, especially in noisy conditions. Its better resilience to noise makes it more suitable than CASIA-WebFace.

2. **Preprocessing Importance**: Given the significant impact of noise, especially Gaussian noise, implementing robust image preprocessing and denoising techniques is crucial for maintaining system performance.

3. **Confidence Thresholds**: As noise increases, the likelihood of false positives and false negatives increases. Dynamic adjustment of confidence thresholds based on detected image quality could help maintain system reliability.

4. **Multi-frame Analysis**: For video-based facial recognition, implementing multi-frame analysis could help mitigate the effects of motion blur and random noise.

5. **Continuous Monitoring**: Regular monitoring of input image quality and system performance metrics can help detect when environmental conditions are leading to decreased recognition accuracy.
